In [1]:
from lib.Forecast.PatchTransformerExoForecast import PatchTransformerExoForecast
from lib.Dataloaders.ECGDataset import ECGDataset
from torch.utils.data import DataLoader

In [2]:
lookback = 100
horizon = 25
batch_size = 2048
stride = 10
norm = "zscore"
norm_all = False
samples = (2500, 200, 200)
channel = 0
conditional = True
exogenous = True

In [3]:
train = ECGDataset(
    dir='../Preprocess/PTBXL',
    dataset='train',
    nsamples=samples[0],
    channel=channel,
    norm=norm,
    norm_all=norm_all,
    lookback=lookback,
    horizon=horizon,
    stride=stride,
labels=True,)

train = DataLoader(train, batch_size=batch_size, shuffle=True)


Loading data from ../Preprocess/PTBXL
train
X_train shape is (220000, 8, 100)
y_train shape is (220000, 25)
dlabels shape is (220000, 8)
NORM=zscore


In [6]:
model = PatchTransformerExoForecast(lookback=lookback,
                                  horizon=horizon,
                                  input_dim=8,
                                  target_dim=1,
                                  condition_dim=8,
                                  d_model=64,
                                  n_heads=8,
                                  num_layers=2,
                                  norm_first=True,
                                  dropout=0,
                                  patch_len=10,
                                  layer_norm_eps=1e-5,
                                  bias=True,
                                  swiglu=True,
                                  rmsnorm=True,
                                  trans_norm=True,
                                  device='cuda',
                                  verbose=False
                                  ).to('cuda')
# model.compile()

In [7]:
for i, data in enumerate(train):
    if conditional:
        lookback_data, horizon_data, cond_data = data
        cond_data = cond_data.to('cuda')
    else:
        lookback_data, horizon_data = data
        cond_data = None
    lookback_data = lookback_data.to('cuda')
    horizon_data = horizon_data.to('cuda')
    # We do not have the exogenous data in the dataset, so we will use the lookback data as exogenous data
    if exogenous:
        exogenous_data = lookback_data.clone()
    exogenous_data = exogenous_data.to('cuda') if exogenous else None

    # take the data and upload it to the GPU
    inputs = lookback_data.to('cuda')
    y = horizon_data.to('cuda')

    print(
        f"Input shape: {inputs.shape}, Exogenous shape: {exogenous_data.shape if exogenous else None}, Conditional shape: {cond_data.shape if conditional else None}"
    )

    outputs = model(
        inputs,
        exo=exogenous_data if exogenous else None,
        cond=cond_data if cond_data is not None else None).squeeze()

    break  # Just run one batch for testing

Input shape: torch.Size([2048, 8, 100]), Exogenous shape: torch.Size([2048, 8, 100]), Conditional shape: torch.Size([2048, 8])
